# 05. Which part of FIMA-Q is fragile

This notebook runs the ablation tests for fragility, it also compares the team's predictions against its result. The base we use is FIMA-Q's W3A3 checkpoint on DeiT-S, with W4A4, W6A6 and W8A8 for comparison. The switch tests use the checkpoints from after calibration, because the finished ones hold rounded weights and cannot return to FP32. A point counts as fragile only if its paired difference from the base is at least 0.5 points on val, with an interval that excludes zero. Same way on V2 and R.

Code: `methods/fimaq/fragility.py` (tests 1 to 5), `methods/fimaq/clipping.py`, and full searches through `methods/fimaq/run.sh --float-points`. With `RERUN = False` everything is read from `outputs/`.

In [ ]:
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

import glob
import os
import subprocess
import sys

REPO = Path.cwd()
THIRD = Path(os.environ.get("THIRD_PARTY", REPO.parent / "third_party"))   # scripts/setup_third_party.sh
PY = sys.executable                                     # this environment runs our scripts
os.environ.setdefault("PY", PY)
os.environ.setdefault("ADALOG_PY", "python")            # the published methods' environment, see the README

def sh(cmd):
    """Run a command from the repository root, show its output as it comes, and stop if it fails."""
    with subprocess.Popen(cmd, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT) as p:
        for line in p.stdout:
            print(line, end="")
    if p.returncode:
        raise RuntimeError(f"exit code {p.returncode}: {cmd}")

def latest(pattern):
    """The newest file under ../third_party that matches `pattern`."""
    found = sorted(glob.glob(str(THIRD / pattern)), key=os.path.getmtime)
    if not found:
        raise FileNotFoundError(pattern)
    return found[-1]

def baselines():
    """The dense baselines of notebook 02, as {model: {"size_mb": ..., "accuracy": ...}}."""
    b = json.loads((Path("outputs") / "baselines_summary.json").read_text())
    return {k: {"size_mb": b["size_mb"][k], "accuracy": a} for k, a in b["accuracy"].items()}

def checkpoint_of_run(name):
    """The checkpoint a FIMA-Q or APHQ-ViT run saved after block reconstruction, read from its log."""
    folder = THIRD / ("APHQ-ViT-protocol" if name.startswith("APHQ") else "FIMA-Q-protocol")
    log = (folder / f"run_{name.replace(' ', '_')}.log").read_text(errors="replace")
    saved = [l.split("Saving checkpoint to ")[1].strip() for l in log.splitlines() if "Saving checkpoint to" in l and "optimsize" in l]
    return str(folder / saved[-1])

RERUN = False          # True calls the scripts; each FIMA-Q search takes 40 min to 2 h
OUT = Path("outputs")
SETS = ["ImageNet-1k val", "ImageNet-V2", "ImageNet-R"]
load = lambda f: json.loads((OUT / f).read_text())
top1 = lambda e, s="ImageNet-1k val": e["accuracy"][s]["top1"]
def row(name, e, ref=None):
    accs = " / ".join(f"{top1(e, s):6.2f}" for s in SETS)
    d = f"   vs ref {top1(e) - top1(ref):+6.2f}" if ref else ""
    size = f"{e['size_mb']:7.2f} MB" if e.get("size_mb") else "      -   "   # points scored in memory have no file
    print(f"{name:44s} {size}  {accs}{d}")

## Predictions

In [ ]:
text = Path("ABLATIONS.md").read_text()
print(text[text.index("## Which part of FIMA-Q is fragile"):])

## Rerunning (optional)

The tests start from the checkpoints of the FIMA-Q runs as published in notebook 03. Each run's folder is the one its entry in `published_summary.json` points to, and holds both the checkpoint after calibration, which the switch tests use, and the finished one, which the clipping counts use. Every script here runs in the method's environment (`ADALOG_PY`).

In [ ]:
if RERUN:
    pub = load("published_summary.json")
    def ckpt(wa, stage):   # stage "calibsize_128_mse" (after calibration) or "optimsize_*" (finished)
        folder = (REPO / pub[f"FIMA-Q {wa}"]["checkpoint"]).parent
        return glob.glob(str(folder / f"deit_small_{wa[:2].lower()}_{wa[2:].lower()}_{stage}.pth"))[0]
    frag = '"$ADALOG_PY" methods/fimaq/fragility.py'
    sh(f'{frag} --config ./configs/3bit/best.py --ckpt {ckpt("W3A3", "calibsize_128_mse")} --tag W3A3cal --tests 1 2 3')
    sh(f'{frag} --config ./configs/4bit/best.py --ckpt {ckpt("W4A4", "calibsize_128_mse")} --tag W4A4cal --tests 1 2 3')
    sh(f'{frag} --config ./configs/6bit/best.py --ckpt {ckpt("W6A6", "calibsize_128_mse")} --tag W6A6cal --tests 1')
    sh(f'{frag} --config ./configs/3bit/best.py --ckpt {ckpt("W3A3", "calibsize_128_mse")} --tag W3A3cal --tests 5 '
       '--points "fc2 input" "attention scores" "qkv input"')
    for b in (3, 4):
        sh(f'"$ADALOG_PY" methods/fimaq/clipping.py --config ./configs/{b}bit/best.py --ckpt {ckpt(f"W{b}A{b}", "optimsize_*")} --tag W{b}A{b}')
    for p in ["fc2", "K", "scores"]:
        sh(f'SCORE_OUT=fragility_search.json bash methods/fimaq/run.sh 3bit/best.py "FIMA-Q W3A3 float {p}" --float-points {p}')
    sh(f'SCORE_OUT=fragility_search.json bash methods/fimaq/run.sh 3bit/best.py "FIMA-Q W3A3 seed1" --calib-list "{REPO}/splits/calib_seed1.txt"')

## Test 1: weights against activations

Each checkpoint with everything quantized (base), only its weights, only its activations, and nothing quantized.

In [ ]:
frag = {f.stem: load(f.name) for f in sorted(OUT.glob("fragility_*cal_*.json"))}
for name, d in frag.items():
    base = next((v for k, v in d.items() if k.endswith("| base")), None)
    print("==", name)
    for k, v in d.items():
        if any(k.endswith(x) for x in ("| base", "| weights only", "| activations only", "| everything off")):
            row(k, v, base)

## Tests 2 and 3: remove one point, keep one point

Every configuration against its checkpoint's base on val. A long bar means that point matters.

In [ ]:
for name in ["fragility_W3A3cal_123", "fragility_W4A4cal_123"]:
    d = frag[name]
    base = next(v for k, v in d.items() if k.endswith("| base"))
    rest = {k.split("| ", 1)[1]: top1(v) - top1(base) for k, v in d.items()
            if not any(k.endswith(x) for x in ("| base", "| weights only", "| activations only", "| everything off"))}
    rest = dict(sorted(rest.items(), key=lambda kv: kv[1]))
    fig, ax = plt.subplots(figsize=(7, 0.25 * len(rest) + 1))
    ax.barh(list(rest), list(rest.values()), color=["#C44E52" if v < 0 else "#55A868" for v in rest.values()])
    ax.axvline(0, c="black", lw=0.8)
    ax.set(xlabel="top-1 on val minus the base (points)", title=name); ax.tick_params(axis="y", labelsize=7)
    plt.tight_layout(); plt.show()

## Paired intervals 

Every configuration of the remove-one and width tests against its checkpoint's base, with the paired difference on val, V2 and R and whether it passes the 0.5 difference. The keep-one configurations are read against FP32, where a point is costly if alone it loses at least 0.5 on val with the same sign on V2 and R.

In [ ]:
from deitc.evaluate import paired_diff_ci
SHORT = {"ImageNet-1k val": "val", "ImageNet-V2": "V2", "ImageNet-R": "R"}

def rule(tag):
    """Each configuration of one fragility file, with the 0.5-point rule applied where it has a reference."""
    j, z = load(f"fragility_{tag}.json"), np.load(OUT / f"fragility_{tag}.npz")
    t = tag.split("_")[0]
    zb = np.load(OUT / f"fragility_{t}_123.npz") if tag.endswith("_5") else z   # test 5 is paired with the base of tests 1 to 3
    get = lambda lab, s: (zb if lab == "base" else z)[f"{s}|{t} | {lab}"].astype(float)
    print(f"\n== {tag} ({len(j)} configurations)")
    for v in j.values():
        lab, a = v["label"], v["accuracy"]
        line = f"{lab:28s} " + " ".join(f"{SHORT[s]} {a[s]['top1']:6.2f} ({a[s]['vs_fp32']['diff']:+6.2f})" for s in SETS)
        if lab.startswith(("remove", "width")):
            ds = [paired_diff_ci(get(lab, s), get("base", s)) for s in SETS]
            line += "  vs base: " + " ".join(f"{d:+5.2f} [{c[0]:+.2f}, {c[1]:+.2f}]" for d, c in ds)
            if ds[0][0] >= 0.5 and ds[0][1][0] > 0 and all(d > 0 for d, _ in ds[1:]):
                line += "  FRAGILE"
        if lab.startswith("keep"):
            cs = [a[s]["vs_fp32"] for s in SETS]
            line += f"  vs FP32 on val [{cs[0]['ci95'][0]:+.2f}, {cs[0]['ci95'][1]:+.2f}]"
            if cs[0]["diff"] <= -0.5 and cs[0]["ci95"][1] < 0 and all(c["diff"] < 0 for c in cs[1:]):
                line += "  COSTLY"
        print(line)

for tag in ["W3A3cal_123", "W4A4cal_123", "W6A6cal_1", "W3A3cal_5"]:
    rule(tag)

In [ ]:
# Do the remove-one recoveries add up to the whole gap between the base and the weights alone?
for t in ["W3A3cal", "W4A4cal"]:
    j = {v["label"]: v["accuracy"] for v in load(f"fragility_{t}_123.json").values()}
    val = lambda lab: j[lab]["ImageNet-1k val"]["top1"]
    recovered = sum(val(lab) - val("base") for lab in j if lab.startswith("remove"))
    print(f"{t}: sum of the remove-one recoveries {recovered:+.2f}, gap from the base to weights only {val('weights only') - val('base'):+.2f}")

## Test 5: how low each point can go

The three points the first tests ranked highest, at A3, A4, A6 and A8, with only that point's scale searched again. Our search is simpler than FIMA-Q's, so A3 through our search shows what the simpler search itself costs.

In [ ]:
d = frag["fragility_W3A3cal_5"]
base = next((v for k, v in d.items() if k.endswith("| base")), None)
for k, v in d.items():
    row(k, v, base)

## Full searches with one point left at full precision

The switch tests show what breaks before block reconstruction. These rerun the whole FIMA-Q search with one point at full precision, against the published W3A3 run, and the second calibration seed shows how much the base itself moves.

In [ ]:
ref = load("published_summary.json")["FIMA-Q W3A3"]
row("FIMA-Q W3A3 (published run, seed 0)", ref)
for k, v in load("fragility_search.json").items():
    row(k, v, ref)

## Under shift: how many activations clip

The share of each point's activations that falls outside its quantizer's range, on val against R. The scales were fitted on training images, so a point whose clipping grows on R is one that distribution shift pushes out of range.

In [ ]:
for tag in ["W3A3", "W4A4"]:
    c = load(f"clipping_{tag}.json")
    pts = list(c["ImageNet-1k val"])
    x = np.arange(len(pts))
    fig, ax = plt.subplots(figsize=(8, 3))
    for i, s in enumerate(SETS):
        ax.bar(x + (i - 1) * 0.27, [100 * c[s][p] for p in pts], 0.27, label=s)
    ax.set_xticks(x, pts, rotation=30, ha="right", fontsize=8)
    ax.set(ylabel="clipped (%)", title=f"FIMA-Q {tag}"); ax.legend(fontsize=7)
    plt.tight_layout(); plt.show()

## Weight sensitivity per layer

The other half of fragility: one unit changed at a time, the rest in FP32, for four quantizers and three pruning methods (`scripts/allocate/sensitivity.py`, `outputs/sensitivity_summary.json`, 890 configurations). The cell below summarises it: the share of the divergence each kind of layer carries, whether the ranking on calibration images holds on val, how far the methods agree, whether the per-layer effects add up, and the pruning sensitivity per block.

In [ ]:
if RERUN:
    sh(f'"{PY}" scripts/allocate/sensitivity.py all')   # about 2 hours; it skips configurations already in the summary
from scipy.stats import spearmanr
P = load("sensitivity_summary.json")["points"]
pub, val = load("published_summary.json"), load("validation_summary.json")
fp32 = top1(baselines()["DeiT-S FP32"])
units = list(P["RTN"]["W3"])
vec = lambda m, st, sp="calib", met="kl", U=units: np.array([P[m][st][u][sp][met] for u in U])
QUANT = [("RTN", "W2"), ("RTN", "W3"), ("RTN", "W4"), ("AdaLog", "W3"), ("FIMAQ", "W3"), ("FIMAQ", "W4"), ("APHQ", "W3")]

print("Share of the KL on calibration images carried by each kind of layer, and by each block")
kinds = {"qkv": "attn.qkv", "proj": "attn.proj", "fc1": "mlp.fc1", "fc2": "mlp.fc2"}
for m, st in QUANT:
    tot = vec(m, st).sum()
    share = {n: round(sum(P[m][st][u]["calib"]["kl"] for u in units if u.endswith(p)) / tot, 3) for n, p in kinds.items()}
    share["embed"] = round(P[m][st]["patch_embed.proj"]["calib"]["kl"] / tot, 3)
    share["head"] = round(P[m][st]["head"]["calib"]["kl"] / tot, 3)
    blocks = [round(sum(P[m][st][u]["calib"]["kl"] for u in units if u.startswith(f"blocks.{i}.")) / tot, 2) for i in range(12)]
    print(f"  {m} {st}", share, "sum of KL", round(tot, 3), "blocks", blocks)

print("\nDoes the ranking on calibration images hold on val? Spearman over layers")
for m, st in QUANT:
    a, b, c, d = vec(m, st), vec(m, st, "val"), vec(m, st, "val", "top1_change"), vec(m, st, "calib", "top1_change")
    print(f"  {m} {st}: KL calib vs KL val {spearmanr(a, b)[0]:.3f}, KL calib vs top-1 change on val {spearmanr(a, c)[0]:.3f}, "
          f"top-1 change calib vs val {spearmanr(d, c)[0]:.3f}")

print("\nDo the methods agree on which layers matter? Spearman on calibration KL at W3")
A = [("RTN", "W3"), ("AdaLog", "W3"), ("FIMAQ", "W3"), ("APHQ", "W3")]
for x in A:
    print(f"  {x[0]:7s}", [round(spearmanr(vec(*x), vec(*y))[0], 3) for y in A])

print("\nDo the per-layer effects add up? Sum of the per-layer top-1 changes on val against the whole model's change")
for n, m, st, whole in [("RTN W3", "RTN", "W3", val["RTN W3 per row, AdaLog's layers"]), ("RTN W4", "RTN", "W4", val["RTN W4 per row, AdaLog's layers"]),
                        ("FIMA-Q W3", "FIMAQ", "W3", pub["FIMA-Q W3"]), ("FIMA-Q W4", "FIMAQ", "W4", pub["FIMA-Q W4"]),
                        ("AdaLog W3", "AdaLog", "W3", pub["AdaLog W3"])]:
    print(f"  {n}: sum of layers {vec(m, st, 'val', 'top1_change').sum():+.2f}, whole model {top1(whole) - fp32:+.2f}")

print("\nLayers where FIMA-Q gains most over round-to-nearest at W3 (calibration KL)")
gain = np.argsort(-(vec("RTN", "W3") - vec("FIMAQ", "W3")))[:8]
print(" ", [(units[i], round(vec("RTN", "W3")[i], 4), round(vec("FIMAQ", "W3")[i], 4)) for i in gain])

print("\nPruning, per block at each sparsity: calibration KL and top-1 change on val")
for m in P:
    if m in ("RTN", "AdaLog", "FIMAQ", "APHQ"):
        continue
    for st in P[m]:
        for unit in ("mlp", "attn") if m == "CORP" else ("mlp",):
            us = [f"blocks.{i}.{unit}" for i in range(12)]
            print(f"  {m} {st} {unit} KL ", [round(P[m][st][u]["calib"]["kl"], 3) for u in us])
            print(f"  {m} {st} {unit} top-1", [round(P[m][st][u]["val"]["top1_change"], 1) for u in us],
                  "sum", round(sum(P[m][st][u]["val"]["top1_change"] for u in us), 1))